<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/day19-22-Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base_layer, r=8, alpha=16):
        super().__init__()
        self.base = base_layer
        in_f  = base_layer.in_features
        out_f = base_layer.out_features

        # 1. 凍結原權重
        for p in self.base.parameters():
            p.requires_grad = False

        # 2. 兩個小矩陣
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, r))

        self.scaling = alpha / r

    def forward(self, x):
        base_out = self.base(x)                    # x @ W₀ᵀ
        lora_out = (x @ self.A.T) @ self.B.T       # x @ Aᵀ @ Bᵀ
        return base_out + lora_out * self.scaling